# El avión de 150 plazas del guion V4, paso a paso

Esta notebook recorre el guion
(`docs/Dimensionamiento_preliminar_aeronave_Raymer_V4.pdf`) sección a sección
con las funciones del código: un birreactor de pasillo único, 150 pasajeros y
3 000 NM, dimensionado con el método de Raymer
(*Aircraft Design: A Conceptual Approach*, 7.ª ed.).

Los títulos y los números de ecuación son los del PDF. Cada celda imprime el
valor del código junto al del guion y el error relativo.

**Para ejecutarla:** `pip install -r requirements-dev.txt` desde la raíz del
repositorio y abre la notebook con Jupyter o VS Code.

In [1]:
import sys
from math import radians, sqrt, degrees
from pathlib import Path

# Los módulos se importan desde la raíz del repositorio
ROOT = Path.cwd() if (Path.cwd() / "main.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


def compara(etiqueta, codigo, guion, unidad="", fmt=",.3f"):
    # Valor del código, valor del guion y error relativo
    print(f"{etiqueta:<34}{codigo:>12{fmt}} {unidad:<6}{guion:>12{fmt}}   {codigo / guion - 1:+6.1%}")


def cabecera():
    print(f"{'':<34}{'código':>12} {'':<6}{'guion':>12}   error")

## 1.3 Requisitos de misión adoptados

Los requisitos de la Tabla 1 van en `Mission`. Todo el código trabaja en SI,
con las **masas en kg**; `core.constants` trae las conversiones `NM`, `KT` y
`HOUR`.

Las fracciones de los segmentos cortos son las de la Tabla 2 (sección 4.1). El
guion junta descenso y aterrizaje en un único 0,995, así que se quita el
descenso con `descent=False`.

In [2]:
from core.constants import NM, KT, HOUR, G
from core.data import Mission

mission = Mission(
    n_pax=150, m_pax=100.0,          # pasajero con equipaje [kg]
    n_trip=6, m_trip=95.0,           # 2 pilotos + 4 TCP
    range=3000 * NM,                 # [m]
    mach=0.78,
    altitude=11_000,                 # FL360 [m]
    loiter=45 * 60,                  # espera [s]
    v_aprox=135 * KT,                # [m/s]
    F_takeoff=0.970, F_ascent=0.985, F_landing=0.995,   # Tabla 2
    descent=False,
    F_reserve=1.06,                  # 6 % de combustible atrapado y reserva
)

## 2–3 Ecuación de pesos, tripulación y carga de pago

$$
W_0 = \frac{W_\text{crew} + W_\text{payload}}{1 - \dfrac{W_f}{W_0} - \dfrac{W_e}{W_0}} \qquad (2)
$$

El numerador es el único término que no depende de $W_0$.

In [3]:
from methods import weight

cabecera()
compara("Wpayload = 150 · 100 (3)", weight.w_payload(mission), 15_000, "kg", ",.0f")
compara("Wcrew = 6 · 95 (4)", weight.w_crew(mission), 570, "kg", ",.0f")
compara("Wcrew + Wpayload", weight.w_fixed(mission), 15_570, "kg", ",.0f")

                                        código              guion   error
Wpayload = 150 · 100 (3)                15,000 kg          15,000    +0.0%
Wcrew = 6 · 95 (4)                         570 kg             570    +0.0%
Wcrew + Wpayload                        15,570 kg          15,570    +0.0%


## 5 Estimación aerodinámica: eficiencia L/D

Breguet (sección 4) necesita la $L/D$, así que las hipótesis aerodinámicas se
definen antes. Van en `Aerodynamics`; la flecha va **en radianes** y los
consumos específicos en **s⁻¹**.

$$
(L/D)_\text{máx} = K_{LD}\sqrt{\frac{A}{S_\text{mojada}/S_\text{ref}}} \quad (12)
\qquad
(L/D)_\text{crucero} = 0{,}866\,(L/D)_\text{máx} \quad (14)
$$

In [4]:
from core.data import Aerodynamics
from methods import aerodynamic

aero = Aerodynamics(
    AR=9.5,                          # alargamiento
    swet_sref=6.0,                   # superficie mojada / de referencia
    k_ld=15.5,                       # reactores civiles (Raymer 3.4.4)
    taper_ratio=0.24,                # sección 8.5
    sweep_c4=radians(25.0),          # flecha en c/4, sección 8.2.1 [rad]
    c_cruise=0.5 / HOUR,             # TSFC de crucero (Tabla 3.3) [1/s]
    c_loiter=0.4 / HOUR,             # TSFC de espera [1/s]
    cfe=0.0026,                      # Tabla 12.3 -> CD0 = Cfe · Swet/Sref
)

cabecera()
compara("A / (Swet/Sref)", aero.AR / aero.swet_sref, 1.58, "", ".2f")
compara("(L/D)máx (13)", aerodynamic.ld_max(aero), 19.5, "", ".2f")
compara("(L/D)crucero (14)", aerodynamic.ld_cruise(aero), 16.9, "", ".2f")
compara("CD0 = Cfe · Swet/Sref", aerodynamic.cd0(aero), 0.016, "", ".4f")

                                        código              guion   error
A / (Swet/Sref)                           1.58               1.58    +0.2%
(L/D)máx (13)                            19.50              19.50    +0.0%
(L/D)crucero (14)                        16.89              16.90    -0.1%
CD0 = Cfe · Swet/Sref                   0.0156             0.0160    -2.5%


El guion redondea $C_{D0}$ a 0,016; el código usa el valor exacto, 0,0156.

## 4 Fracción de peso de combustible

### 4.2 Crucero: Breguet del alcance, con $V = M\,a$

$$
\frac{W_{i+1}}{W_i} = \exp\left(-\frac{R\,c}{V\,(L/D)}\right) \qquad (6)
$$

### 4.3 Reservas: Breguet de la autonomía, en $(L/D)_\text{máx}$

$$
\frac{W_{i+1}}{W_i} = \exp\left(-\frac{E\,c}{L/D}\right) \qquad (9)
$$

### 4.4 Fracción total

$$
\frac{W_x}{W_0} = \prod \frac{W_{i+1}}{W_i} \quad (10)
\qquad
\frac{W_f}{W_0} = 1{,}06\left(1 - \frac{W_x}{W_0}\right) \quad (11)
$$

In [5]:
f_cruise = weight.F_cruise(mission, aero)
f_loiter = weight.F_loiter(mission, aero)
wx_w0 = mission.F_takeoff * mission.F_ascent * f_cruise * f_loiter * mission.F_landing

cabecera()
compara("V = M · a", aerodynamic.velocity_rel(mission), 230, "m/s", ".1f")
compara("W/W crucero (7)", f_cruise, 0.820)
compara("W/W espera (9)", f_loiter, 0.985)
compara("Wx/W0 (10)", wx_w0, 0.768)
compara("Wf/W0 (11)", weight.F_fuel(mission, aero), 0.246)

                                        código              guion   error
V = M · a                                230.2 m/s          230.0    +0.1%
W/W crucero (7)                          0.820              0.820    +0.0%
W/W espera (9)                           0.985              0.985    -0.0%
Wx/W0 (10)                               0.768              0.768    -0.0%
Wf/W0 (11)                               0.246              0.246    +0.1%


## 6 Fracción de peso en vacío

Tabla 3.1 de Raymer, *jet transport* en unidades métricas:

$$
\frac{W_e}{W_0} = 1{,}202\,W_0^{-0{,}072}\,K_{vs} \qquad (16)
$$

## 7 Iteración y cálculo de $W_0$

`weight.resolve` itera la ecuación (2) y devuelve un `Result` con $W_0$, las
fracciones y los pesos absolutos. Sin `refined=True` usa el método de primer
orden de las secciones 4–6. Necesita un `Design`; de momento basta con el
`k_vs` del ala de flecha fija, y el resto se completa en la sección 8.

In [6]:
from core.data import Design

first = weight.resolve(mission, aero, Design(k_vs=1.0))

cabecera()
compara("We/W0 (16)", weight.F_empty(first.w0), 0.537)
compara("W0 (17)", first.w0, 72_000, "kg", ",.0f")
compara("We", first.w_empty, 38_600, "kg", ",.0f")
compara("Wf", first.w_fuel, 17_700, "kg", ",.0f")

                                        código              guion   error
We/W0 (16)                               0.537              0.537    +0.1%
W0 (17)                                 71,940 kg          72,000    -0.1%
We                                      38,652 kg          38,600    +0.1%
Wf                                      17,718 kg          17,700    +0.1%


### 7.1 Validación

El avión de referencia del segmento tiene un MTOW de ~78 000 kg.

In [7]:
print(f"W0 = {first.w0:,.0f} kg frente a 78 000 kg: {first.w0 / 78_000 - 1:+.1%}")

W0 = 71,940 kg frente a 78 000 kg: -7.8%


## 8 Initial sizing: carga alar, superficie y geometría del ala

Las decisiones de proyecto de las secciones 8–10 (carga alar, empuje, cabina,
fuselaje y colas) van en `Design`. La cabina de la sección 9 se describe con
`Deck`, formada por `SeatingZone` (una clase de asientos); los coeficientes de
cola con `TailCoefficients`, y los mandos con `ControlSurfaceRatios`.

In [8]:
from geometry.fuselage import Deck, SeatingZone
from geometry.tail import TailCoefficients, ControlSurfaceRatios

cabin = Deck(                        # sección 9.2, Tabla 5
    name="main",
    zones=[
        SeatingZone("business", n_seats=12, seats_abreast=4, seat_pitch=0.97),   # 2-2
        SeatingZone("economy", n_seats=138, seats_abreast=6, seat_pitch=0.81),   # 3-3
    ],
    aisles=1,
    pax_per_lavatory=50,
)

design = Design(
    wing_loading=600.0,              # 8.3: W0/S [kg/m2]
    cruise_wing_loading=550.0,       # 8.8: W/S supuesta en crucero [kg/m2]
    thrust_to_weight=0.30,           # 8.6
    n_engines=2,
    max_mach=0.82,                   # 1.3: Mach máximo operativo
    cl_max_landing=2.8,              # 8.1
    mlw_fraction=0.85,               # 8.1: peso de aterrizaje / W0
    vref_factor=1.23,                # 8.1: CS-25
    k_vs=1.0,                        # 6: ala de flecha fija
    raymer_edition=7,                # tablas 3.1, 6.1 y 6.3 de la 7.ª edición
    decks=[cabin],
    fuselage={"diameter": 3.95, "nose": 4.0, "tailcone": 5.0},   # 9.1 y Tabla 5 [m]
    tail=TailCoefficients(c_ht=1.00, c_vt=0.09, arm_fraction=0.50),   # 10.1-10.3
    controls=ControlSurfaceRatios(aileron_area_fraction=0.05,         # 10.4, Tabla 6
                                  elevator_chord=0.25, rudder_chord=0.32),
    tail_arm_length="statistical",   # 9.2.1: brazo sobre la longitud de la Tabla 6.3
)

### 8.1 Restricción de aterrizaje

Con $V_\text{aprox} = 1{,}23\,V_\text{stall}$ y $C_{L,\text{máx}} = 2{,}8$, la
carga alar máxima al peso de aterrizaje ($0{,}85\,W_0$) es

$$
\frac{W}{S} = \frac{1}{2}\,\rho\,V_\text{stall}^2\,C_{L,\text{máx}} \qquad (19)
$$

`constraints.landing_wing_loading` la devuelve ya referida al despegue.

In [9]:
from methods import constraints

ws_landing = constraints.landing_wing_loading(mission, design)

cabecera()
compara("V_stall = V_aprox / 1,23", mission.v_aprox / design.vref_factor, 56.5, "m/s", ".1f")
compara("W/S al aterrizaje", ws_landing * design.mlw_fraction, 557, "kg/m2", ",.0f")
compara("W/S máx. al despegue (19)", ws_landing, 655, "kg/m2", ",.0f")

                                        código              guion   error
V_stall = V_aprox / 1,23                  56.5 m/s           56.5    -0.1%
W/S al aterrizaje                          557 kg/m2          557    +0.1%
W/S máx. al despegue (19)                  656 kg/m2          655    +0.1%


### 8.2 Restricción de crucero y 8.2.1 factor de Oswald

$$
C_{L,\text{ópt}} = \sqrt{\frac{C_{D0}}{3K}}, \quad K = \frac{1}{\pi A e} \quad (20)
\qquad
e = 1{,}78\,(1 - 0{,}045\,A^{0{,}68}) - 0{,}64 \quad (21)
$$

La ecuación (21) es la de ala recta, válida porque $\Lambda_\text{BA} < 30°$.

In [10]:
q = 0.5 * aerodynamic.cruise_density(mission) * aerodynamic.velocity_rel(mission) ** 2
cl_opt = sqrt(aerodynamic.cd0(aero) / (3 * aerodynamic.k(aero)))

cabecera()
compara("Λ borde de ataque", degrees(aerodynamic.sweep_le(aero)), 28, "°", ".1f")
compara("Oswald e (21)", aerodynamic.oswald(aero), 0.77)
compara("K = 1/(π A e)", aerodynamic.k(aero), 0.0435, "", ".4f")
compara("CL óptimo (20)", cl_opt, 0.35)
compara("q crucero", q, 9_630, "N/m2", ",.0f")
compara("W/S óptima, al despegue", constraints.cruise_wing_loading(mission, aero), 365, "kg/m2", ",.0f")

                                        código              guion   error
Λ borde de ataque                         28.0 °             28.0    -0.1%
Oswald e (21)                            0.770              0.770    -0.0%
K = 1/(π A e)                           0.0435             0.0435    +0.1%
CL óptimo (20)                           0.346              0.350    -1.2%
q crucero                                9,667 N/m2         9,630    +0.4%
W/S óptima, al despegue                    356 kg/m2          365    -2.3%


La W/S óptima sale 356 kg/m² en lugar de 365 porque el guion redondea $C_{D0}$
a 0,016.

### 8.3 Selección de la carga alar

El aterrizaje admite hasta ~655 kg/m² y el crucero óptimo pediría ~365 kg/m².
Como los aviones reales, el guion se queda cerca del límite alto (ala pequeña,
más ligera y con menos resistencia) y deja margen por debajo del aterrizaje:
$W/S = 600$ kg/m² (22) es una **decisión de proyecto**, `design.wing_loading`.

### 8.4–8.5 Superficie y geometría del ala

$$
S = \frac{W_0}{W/S} \quad (23) \qquad
b = \sqrt{A\,S} \quad (24) \qquad
c_r = \frac{2S}{b\,(1+\lambda)} \quad (25) \qquad
MAC = \frac{2}{3}\,c_r\,\frac{1+\lambda+\lambda^2}{1+\lambda} \quad (26)
$$

In [11]:
from geometry import wing

w1 = wing.wing_geometry(first.w0, aero, design)

cabecera()
compara("S (23)", w1["S"], 120, "m2", ".1f")
compara("b (24)", w1["b"], 33.7, "m", ".2f")
compara("c_raíz (25)", w1["c_root"], 5.74, "m", ".2f")
compara("MAC (26)", w1["MAC"], 4.00, "m", ".2f")

                                        código              guion   error
S (23)                                   119.9 m2           120.0    -0.1%
b (24)                                   33.75 m            33.70    +0.1%
c_raíz (25)                               5.73 m             5.74    -0.2%
MAC (26)                                  4.00 m             4.00    -0.1%


### 8.6 Relación empuje-peso

La Tabla 5.3 da $T/W = 0{,}297\,M_\text{máx}^{0{,}350}$; la subida con un motor
inoperativo lo sube y se elige $T/W = 0{,}30$ (27), `design.thrust_to_weight`.

In [12]:
thrust = design.thrust_to_weight * first.w0 * G / 1e3

cabecera()
compara("T/W estadístico (Tabla 5.3)", constraints.statistical_thrust_to_weight(design), 0.277)
compara("T total (28)", thrust, 212, "kN", ".0f")
compara("T por motor", thrust / design.n_engines, 106, "kN", ".0f")

                                        código              guion   error
T/W estadístico (Tabla 5.3)              0.277              0.277    +0.0%
T total (28)                               212 kN             212    -0.1%
T por motor                                106 kN             106    -0.1%


### 8.7 Fracción en vacío refinada (Tabla 6.1)

$$
\frac{W_e}{W_0} = a\,W_0^{C_1}A^{C_2}\left(\frac{T}{W_0}\right)^{C_3}
\left(\frac{W_0}{S}\right)^{C_4} M_\text{máx}^{C_5}\,K_{vs} \qquad (29)
$$

con los coeficientes en unidades fps; el código hace la conversión.

### 8.8 L/D de crucero con la polar

$$
\frac{L}{D} = \frac{1}{\dfrac{q\,C_{D0}}{W/S} + \dfrac{W}{S}\,\dfrac{1}{q\,\pi A e}}
$$

con $W/S = 550$ kg/m² en crucero (`design.cruise_wing_loading`). El guion la
multiplica por 0,866 en Breguet.

### 8.9 Fracción de ascenso refinada

$$
\frac{W_\text{ascenso}}{W_0} = 1{,}0065 - 0{,}0325\,M \qquad (31)
$$

In [13]:
ld_polar = aerodynamic.ld_cruise_refined(aero, mission, design)

cabecera()
compara("We/W0 Tabla 6.1 (30)", weight.F_empty_refined(first.w0, aero, design), 0.55)
compara("L/D polar", ld_polar, 18.9, "", ".2f")
compara("W/W crucero, 0,866 L/D", weight.breguet_cruise(mission, aero, 0.866 * ld_polar), 0.814)
compara("W/W ascenso (31)", weight.F_ascent_refined(mission), 0.981)

                                        código              guion   error
We/W0 Tabla 6.1 (30)                     0.553              0.550    +0.5%
L/D polar                                19.14              18.90    +1.3%
W/W crucero, 0,866 L/D                   0.817              0.814    +0.4%
W/W ascenso (31)                         0.981              0.981    +0.0%


### 8.10–8.12 Fracción de combustible y $W_0$ refinados

Despegue, espera y aterrizaje mantienen sus fracciones (8.10). Con las nuevas
de crucero y ascenso, y la fracción en vacío de la Tabla 6.1, se vuelve a
iterar la ecuación (2). `guion_refined_w0`, del caso de este ejemplo, repite
esos pasos del guion.

> `weight.resolve(..., refined=True)` no aplica el 0,866 sobre la $L/D$ de la
> polar, que ya es la de crucero (Raymer 6.3.7), y da unos 71 900 kg. El
> detalle está en `example/README.md`.

In [14]:
from example.a320_guion_v4 import guion_refined_w0

w0, wf_w0 = guion_refined_w0(mission, aero, design, first.w0)

cabecera()
compara("Wx/W0", 1 - wf_w0 / mission.F_reserve, 0.759)
compara("Wf/W0", wf_w0, 0.255)
compara("W0 (32)", w0, 79_800, "kg", ",.0f")

                                        código              guion   error
Wx/W0                                    0.762              0.759    +0.4%
Wf/W0                                    0.253              0.255    -1.0%
W0 (32)                                 79,255 kg          79,800    -0.7%


Tabla 4: comparación de los dos métodos con el avión de referencia.

In [15]:
print(f"{'Método':<28}{'We/W0':>8}{'W0 [kg]':>10}{'Δ vs real':>11}")
for name, we_w0, w in [("Simple (Tabla 3.1)", weight.F_empty(first.w0), first.w0),
                       ("Refinado (Tabla 6.1)", weight.F_empty_refined(w0, aero, design), w0)]:
    print(f"{name:<28}{we_w0:>8.2f}{w:>10,.0f}{w / 78_000 - 1:>+11.1%}")
print(f"{'Avión de referencia real':<28}{0.54:>8.2f}{78_000:>10,}")

Método                         We/W0   W0 [kg]  Δ vs real
Simple (Tabla 3.1)              0.54    71,940      -7.8%
Refinado (Tabla 6.1)            0.55    79,255      +1.6%
Avión de referencia real        0.54    78,000


El $W_0$ refinado se adopta como peso de diseño, y el ala y el empuje se
recalculan con él (33)–(34).

In [16]:
w2 = wing.wing_geometry(w0, aero, design)

cabecera()
compara("S (33)", w2["S"], 133, "m2", ".1f")
compara("b", w2["b"], 35.55, "m", ".2f")
compara("c_raíz", w2["c_root"], 6.03, "m", ".2f")
compara("MAC", w2["MAC"], 4.21, "m", ".2f")
compara("T por motor (34)", design.thrust_to_weight * w0 * G / design.n_engines / 1e3, 117, "kN", ".0f")

                                        código              guion   error
S (33)                                   132.1 m2           133.0    -0.7%
b                                        35.42 m            35.55    -0.4%
c_raíz                                    6.01 m             6.03    -0.3%
MAC                                       4.20 m             4.21    -0.3%
T por motor (34)                           117 kN             117    -0.3%


## 9 Dimensionamiento del fuselaje

### 9.1 Sección transversal

Seis asientos por fila (3-3) y un pasillo fijan el ancho de cabina. El
diámetro exterior de 3,95 m es una decisión de proyecto para alojar
contenedores LD3-45 en bodega (`design.fuselage["diameter"]`).

In [17]:
from geometry import fuselage

cabecera()
compara("Ancho interior de cabina", fuselage.cabin_width(cabin), 3.57, "m", ".2f")
compara("Diámetro mínimo", fuselage.external_width(cabin), 3.8, "m", ".2f")

                                        código              guion   error
Ancho interior de cabina                  3.57 m             3.57    +0.0%
Diámetro mínimo                           3.77 m             3.80    -0.8%


### 9.2 Longitud por disposición de cabina (Tabla 5)

`deck_length` suma filas y servicios de la cubierta.

In [18]:
table_5 = {"seating": 2.90 + 18.70, "galleys": 1.80, "lavatories": 2.85,
           "exits": 2.20, "total": 28.45}

cabecera()
for item, length in fuselage.deck_length(cabin).items():
    if item in table_5:
        compara(item, length, table_5[item], "m", ".2f")

                                        código              guion   error
seating                                  21.54 m            21.60    -0.3%
galleys                                   1.80 m             1.80    +0.0%
lavatories                                3.80 m             2.85   +33.3%
exits                                     2.20 m             2.20    +0.0%
total                                    29.34 m            28.45    +3.1%


La cabina sale 0,95 m más larga: el código redondea los lavabos por clase
(1 en preferente + 3 en turista, uno cada 50 pasajeros) y el guion pone 3
para las 150 plazas.

$$
L_f \approx L_\text{cabina} + 4{,}0 + 5{,}0 \quad (35) \qquad
\frac{L_f}{D} \quad (36)
$$

### 9.2.1 Comprobación estadística (Tabla 6.3)

$$
L_f = 0{,}69\,W_0^{0{,}36}
$$

Por coherencia con el método, el guion usa la longitud estadística para el
brazo de cola (`design.tail_arm_length = "statistical"`).

In [19]:
f = fuselage.fuselage_geometry(design.decks, w0=w0, edition=design.raymer_edition,
                               **design.fuselage)

cabecera()
compara("Lf por cabina (35)", f["length"], 37.5, "m", ".2f")
compara("Esbeltez Lf/D (36)", f["fineness"], 9.5, "", ".2f")
compara("Lf estadística (Tabla 6.3)", f["statistical_length"], 40.14, "m", ".2f")

                                        código              guion   error
Lf por cabina (35)                       38.34 m            37.50    +2.2%
Esbeltez Lf/D (36)                        9.71               9.50    +2.2%
Lf estadística (Tabla 6.3)               40.04 m            40.14    -0.2%


## 10 Empenaje y superficies de mando

Brazo de cola $L_{HT} = L_{VT} \approx 0{,}5\,L_f$ (10.1) y coeficientes de
volumen de la Tabla 6.4:

$$
S_{HT} = \frac{c_{HT}\,MAC\,S}{L_{HT}} \quad (38) \qquad
S_{VT} = \frac{c_{VT}\,b\,S}{L_{VT}} \quad (40)
$$

Las superficies de mando (10.4, Tabla 6) son fracciones de la superficie sobre
la que actúan: alerones el 5 % de $S$, timón de profundidad el 25 % de
$S_{HT}$ y timón de dirección el 32 % de $S_{VT}$.

In [20]:
from geometry import tail

t = tail.tail_geometry(w2, f["statistical_length"], design.tail, design.controls)
c = t["controls"]

cabecera()
compara("Brazo de cola", t["arm"], 20.07, "m", ".2f")
compara("S_HT (39)", t["s_ht"], 27.9, "m2", ".1f")
compara("S_VT (41)", t["s_vt"], 21.2, "m2", ".1f")
compara("Alerones", c["ailerons"], 6.6, "m2", ".1f")
compara("Timón de profundidad", c["elevator"], 7.0, "m2", ".1f")
compara("Timón de dirección", c["rudder"], 6.8, "m2", ".1f")

                                        código              guion   error
Brazo de cola                            20.02 m            20.07    -0.2%
S_HT (39)                                 27.7 m2            27.9    -0.8%
S_VT (41)                                 21.0 m2            21.2    -0.8%
Alerones                                   6.6 m2             6.6    +0.1%
Timón de profundidad                       6.9 m2             7.0    -1.1%
Timón de dirección                         6.7 m2             6.8    -1.0%


## 11 Resumen de resultados y comparación (Tabla 7)

Los valores del código frente a los del guion y a la aeronave de referencia.

In [21]:
summary = [
    ("W0 refinado [kg]", w0, 79_800, 78_000, ",.0f"),
    ("Combustible [kg]", wf_w0 * w0, 20_350, 18_700, ",.0f"),
    ("Superficie alar [m2]", w2["S"], 133, 123, ".1f"),
    ("Envergadura [m]", w2["b"], 35.5, 34.0, ".2f"),
    ("Longitud de fuselaje [m]", f["statistical_length"], 40.14, 37.6, ".2f"),
    ("Diámetro de fuselaje [m]", design.fuselage["diameter"], 3.95, 3.95, ".2f"),
    ("Estab. horizontal [m2]", t["s_ht"], 27.9, 31.0, ".1f"),
    ("Estab. vertical [m2]", t["s_vt"], 21.2, 21.5, ".1f"),
    ("Empuje por motor [kN]", design.thrust_to_weight * w0 * G / design.n_engines / 1e3, 117, 110, ".0f"),
]
print(f"{'Parámetro':<26}{'código':>10}{'guion':>10}{'referencia':>12}{'Δ':>9}")
for name, value, guion, ref, fmt in summary:
    print(f"{name:<26}{value:>10{fmt}}{guion:>10{fmt}}{ref:>12{fmt}}{value / ref - 1:>+9.1%}")

Parámetro                     código     guion  referencia        Δ
W0 refinado [kg]              79,255    79,800      78,000    +1.6%
Combustible [kg]              20,013    20,350      18,700    +7.0%
Superficie alar [m2]           132.1     133.0       123.0    +7.4%
Envergadura [m]                35.42     35.50       34.00    +4.2%
Longitud de fuselaje [m]       40.04     40.14       37.60    +6.5%
Diámetro de fuselaje [m]        3.95      3.95        3.95    +0.0%
Estab. horizontal [m2]          27.7      27.9        31.0   -10.7%
Estab. vertical [m2]            21.0      21.2        21.5    -2.2%
Empuje por motor [kN]            117       117         110    +6.0%


Δ es la diferencia del código frente a la referencia. El empuje de referencia
es el centro de la banda de 100–120 kN por motor.